In [1]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

# BB84 Quantum Key Distribution — With Attacker (Eve)

This notebook simulates BB84 **in the presence of an eavesdropper, Eve**, who performs an **intercept-resend attack**: she intercepts each qubit Alice sends, measures it in a randomly chosen basis, and resends a new qubit reflecting her measurement result.

## Why Eve is detectable

The **No-Cloning Theorem** prevents Eve from copying a qubit. When she measures in the *wrong* basis, she destroys the original state and resends a disturbed qubit. This introduces errors in Bob's results even at positions where Alice and Bob used the **same** basis. Alice and Bob detect this by publicly comparing a **sample** of their shared key bits — a high error rate reveals Eve.

## Attack detection threshold

A random intercept-resend attack causes ~25% errors in the matching-basis bits (Eve guesses wrong basis 50% of the time, and when she does, Bob's result is random). Alice and Bob sacrifice a sample of key bits for checking; if the error rate exceeds a threshold (here **15%**), they conclude the channel is compromised.

In [2]:
simulator = BasicSimulator()

# Number of qubits Alice sends
N = 50

# If error rate exceeds this, they abort
ERROR_THRESHOLD  = 0.15

print(f"N={N} qubits, error threshold={ERROR_THRESHOLD}")

N=50 qubits, error threshold=0.15


## Quantum Random Number Generation

Below function is generated via preparing *n* qubits in the state $|+\rangle = \frac{1}{\sqrt{2}}(|0\rangle + |1\rangle)$ and measuring each one. Each measurement collapses to $|0\rangle$ or $|1\rangle$ with probability of $\frac{1}{2}$

In [3]:
def quantum_random_bit():
  """
  Generate one truly random bit by preparing |+⟩ = (|0⟩+|1⟩)/√2
  and measuring it.
  """
  qc = QuantumCircuit(1, 1)
  qc.h(0)
  qc.measure(0, 0)
  compiled = transpile(qc, simulator)
  result = simulator.run(compiled, shots=1).result()
  return int(list(result.get_counts().keys())[0])

def quantum_random_bits(n):
  """Generate n random bits by calling quantum_random_bit() n times."""
  return [quantum_random_bit() for _ in range(n)]

# Quick sanity check
sample = quantum_random_bits(10)
print("Sample quantum random bits:", sample)

Sample quantum random bits: [0, 1, 1, 0, 1, 0, 0, 1, 0, 1]


## Alice Part

Alice generates random bits and random basis choices, then encodes each bit as a qubit in her chosen basis:

- Standard basis (basis=0):  $0 \rightarrow |0\rangle$,  $1 \rightarrow |1\rangle$
- Diagonal basis (basis=1):  $0 \rightarrow |+\rangle$,  $1 \rightarrow |-\rangle$

In [4]:
alice_bits  = quantum_random_bits(N)
alice_bases = quantum_random_bits(N)

def alice_prepare_qubit(bit, basis):
    """
    Prepare a qubit encoding `bit` in `basis`.

    Standard (basis=0):  0 → |0⟩ (no gate)   1 → |1⟩ (X gate)
    Diagonal (basis=1):  0 → |+⟩ (H gate)    1 → |−⟩ (X then H)
    """
    qc = QuantumCircuit(1, 1)
    if bit == 1:
        qc.x(0)
    if basis == 1:
        qc.h(0)
    return qc

# Alice sends qubits into the quantum channel (where Eve is waiting)
alice_channel = [alice_prepare_qubit(alice_bits[i], alice_bases[i]) for i in range(N)]

print("=== ALICE ===")
print(f"Bits:  {alice_bits[:20]} ... (showing first 20)")
print(f"Bases: {['S' if b==0 else 'D' for b in alice_bases[:20]]} ...")
print(f"\nAlice has prepared {N} qubits and sent them into the channel.")

=== ALICE ===
Bits:  [0, 0, 0, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1] ... (showing first 20)
Bases: ['S', 'D', 'D', 'S', 'S', 'D', 'S', 'S', 'D', 'S', 'S', 'D', 'S', 'D', 'D', 'D', 'D', 'S', 'S', 'S'] ...

Alice has prepared 50 qubits and sent them into the channel.


## Eve (Attacker) — Intercept-Resend Attack

Eve intercepts every qubit Alice sends.

For each qubit:
1. A random measurement basis is selected (Alice's choice is unknown).
2. Qubit in that basis is measured — this irreversibly collapses the quantum state (the No-Cloning Theorem prevents copying of qubit while keeping the original).
3. A fresh qubit is prepared to match the her measurement result and forwards it to Bob.


### Outcomes
When Eve guesses the **SAME** basis as Alice: she gets the right bit and Bob receives the correct state — no disruption.

When Eve guesses the **WRONG** basis: her measurement gives a random result and she forwards a disturbed state. If Alice and Bob happen to use the same basis for that qubit, Bob will still get the wrong bit ~50% of those cases → overall ~25% error rate in the shared key.

In [5]:
eve_bases = quantum_random_bits(N)

def eve_intercept_and_resend(qubit_circuit, eve_basis):
    """
    Eve intercepts `qubit_circuit`, measures in `eve_basis`,
    and returns a freshly prepared qubit based on her result.

    This simulates the No-Cloning constraint: Eve must measure
    (destroying the original state) before she can resend anything.
    """
    # Eve Measure
    qc = qubit_circuit.copy()
    if eve_basis == 1:
        qc.h(0)
    qc.measure(0, 0)
    compiled = transpile(qc, simulator)
    result = simulator.run(compiled, shots=1).result()
    eve_bit = int(list(result.get_counts().keys())[0])

    # Resends a fresh qubit matching her measurement
    new_qc = QuantumCircuit(1, 1)
    if eve_bit == 1:
        new_qc.x(0)
    if eve_basis == 1:
        new_qc.h(0)
    return new_qc

# Eve sits on the channel and tampers with all N qubits
eve_channel = [eve_intercept_and_resend(alice_channel[i], eve_bases[i]) for i in range(N)]

print("=== EVE ===")
print(f"Bases: {['S' if b==0 else 'D' for b in eve_bases[:20]]} ...")
print(f"\nEve has intercepted all {N} qubits, measured them, and resent fresh qubits to Bob.")

=== EVE ===
Bases: ['S', 'S', 'D', 'S', 'D', 'S', 'D', 'D', 'S', 'D', 'S', 'D', 'D', 'S', 'D', 'D', 'D', 'S', 'D', 'D'] ...

Eve has intercepted all 50 qubits, measured them, and resent fresh qubits to Bob.


## Bob Part

Bob can potentially receives the tampered qubit from the channel. As he is unaware of Eve (attacker). He picks a random measurement basis for each qubit, just as in the no-attacker scenario.

Diagonal basis measurement: apply H, then measure standard. H maps $|+\rangle \rightarrow |0\rangle$

In [6]:
bob_bases = quantum_random_bits(N)

def bob_measure_qubit(qubit_circuit, basis):
    """Bob measures the received qubit in his chosen basis."""
    qc = qubit_circuit.copy()
    if basis == 1:
        qc.h(0)
    qc.measure(0, 0)
    compiled = transpile(qc, simulator)
    result = simulator.run(compiled, shots=1).result()
    return int(list(result.get_counts().keys())[0])

# Bob measures from Eve's tampered channel (he thinks it's Alice's)
bob_bits = [bob_measure_qubit(eve_channel[i], bob_bases[i]) for i in range(N)]

print("=== BOB ===")
print(f"Bases: {['S' if b==0 else 'D' for b in bob_bases[:20]]} ...")
print(f"Bits:  {bob_bits[:20]} ...")

=== BOB ===
Bases: ['D', 'D', 'S', 'D', 'D', 'S', 'S', 'S', 'S', 'D', 'S', 'S', 'D', 'S', 'D', 'S', 'S', 'S', 'S', 'S'] ...
Bits:  [0, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1] ...


## Public Channel: Basis Sifting

Alice and Bob will compare their basis choices. They keep only the positions where same basis is used.

Eve may also observe this exchange, but it gives her no advantage.

In [7]:
matching_indices = [i for i in range(N) if alice_bases[i] == bob_bases[i]]

alice_sifted = [alice_bits[i] for i in matching_indices]
bob_sifted   = [bob_bits[i]   for i in matching_indices]

print("=== PUBLIC CHANNEL: BASIS COMPARISON ===")
print(f"Total qubits    : {N}")
print(f"Basis matches   : {len(matching_indices)}")
print(f"Sifted key len  : {len(alice_sifted)} bits")

=== PUBLIC CHANNEL: BASIS COMPARISON ===
Total qubits    : 50
Basis matches   : 24
Sifted key len  : 24 bits


## Attack Detection: Error Rate Check

Alice and Bob publicly compare a SAMPLE of their sifted bits. In the absence of Eve, these should always agree. With Eve, ~25% will disagree (on average).

The sample bits are sacrificed (not used in the final key), so the sample size is a trade-off between detection confidence and final key length.

Quantum Randomness is used to decide which sifted bits are sacrificed for checking if there is an attack.

In [8]:
n_sifted      = len(alice_sifted)

# Use quantum randomness to decide which sifted bits to sacrifice for checking
mask = quantum_random_bits(n_sifted)

sample_indices = [i for i, m in enumerate(mask) if m == 1]
key_indices    = [i for i, m in enumerate(mask) if m == 0]

# Compare sample bits over public channel
alice_sample = [alice_sifted[i] for i in sample_indices]
bob_sample   = [bob_sifted[i]   for i in sample_indices]

n_sample     = len(alice_sample)
errors       = sum(a != b for a, b in zip(alice_sample, bob_sample))
error_rate   = errors / n_sample if n_sample > 0 else 0

print("=== ATTACK DETECTION ===")
print(f"Sample size     : {n_sample} bits")
print(f"Errors found    : {errors}")
print(f"Error rate      : {error_rate:.1%}")
print(f"Threshold       : {ERROR_THRESHOLD:.1%}")
print()

attack_detected = error_rate > ERROR_THRESHOLD

if attack_detected:
    print(f"ATTACK DETECTED! Error rate {error_rate:.1%} exceeds threshold {ERROR_THRESHOLD:.1%}.")
    print("   Alice and Bob ABORT — the channel is compromised.")
    print("   (Eve's intercept-resend attack introduced errors because the No-Cloning Theorem prevented her from copying qubits without disturbing them.")
else:
    print(f"No attack detected. Error rate {error_rate:.1%} is within threshold.")
    print("   (This could happen by chance if the sample was small — larger N and larger sample fractions give more reliable detection.)")

=== ATTACK DETECTION ===
Sample size     : 14 bits
Errors found    : 5
Error rate      : 35.7%
Threshold       : 15.0%

ATTACK DETECTED! Error rate 35.7% exceeds threshold 15.0%.
   Alice and Bob ABORT — the channel is compromised.
   (Eve's intercept-resend attack introduced errors because the No-Cloning Theorem prevented her from copying qubits without disturbing them.


## Key Extraction (if not aborted)

If Alice and Bob decide to proceed (no attack detected, or for demonstration purposes), they use the remaining sifted bits (those not sacrificed for sampling) as their shared key.

*Note: with Eve present, these remaining bits will contain ~25% errors, making the key unreliable — which is why aborting is correct.*

In [9]:
alice_key = [alice_sifted[i] for i in key_indices]
bob_key   = [bob_sifted[i]   for i in key_indices]

key_errors    = sum(a != b for a, b in zip(alice_key, bob_key))
key_error_rate = key_errors / len(alice_key) if alice_key else 0

print("=== KEY COMPARISON (for analysis) ===")
print(f"Remaining key length : {len(alice_key)} bits")
print(f"Key errors           : {key_errors}  ({key_error_rate:.1%} error rate)")
print()
print("Alice's key:", alice_key[:30], "..." if len(alice_key) > 30 else "")
print("Bob's key:  ", bob_key[:30],   "..." if len(bob_key) > 30 else "")
print()
print(f"Note: the {key_error_rate:.1%} key error rate confirms Eve's intercept-resend attack was active.")
print("Alice and Bob should have aborted — their 'shared key' is corrupted.")

=== KEY COMPARISON (for analysis) ===
Remaining key length : 10 bits
Key errors           : 2  (20.0% error rate)

Alice's key: [0, 1, 1, 1, 1, 0, 1, 0, 0, 0] 
Bob's key:   [1, 1, 1, 1, 1, 0, 0, 0, 0, 0] 

Note: the 20.0% key error rate confirms Eve's intercept-resend attack was active.
Alice and Bob should have aborted — their 'shared key' is corrupted.


## Summary

In [10]:
print("===== BB84 PROTOCOL SUMMARY (With Attacker) =====")
print(f"  Qubits sent by Alice         : {N}")
print(f"  Basis matches (sifted)       : {len(matching_indices)}")
print(f"  Sample bits checked          : {n_sample}")
print(f"  Sample error rate            : {error_rate:.1%}  (expected ~25% with Eve)")
print(f"  Attack detected              : {attack_detected}")
print(f"  Key error rate (if used)     : {key_error_rate:.1%}")
print()
print("Security guarantee: the No-Cloning Theorem ensures Eve cannot intercept qubits without disturbing them, making her presence detectable.")

===== BB84 PROTOCOL SUMMARY (With Attacker) =====
  Qubits sent by Alice         : 50
  Basis matches (sifted)       : 24
  Sample bits checked          : 14
  Sample error rate            : 35.7%  (expected ~25% with Eve)
  Attack detected              : True
  Key error rate (if used)     : 20.0%

Security guarantee: the No-Cloning Theorem ensures Eve cannot intercept qubits without disturbing them, making her presence detectable.
